<a href="https://colab.research.google.com/github/anupbagale/covid-xray-shortcut-learning/blob/main/notebooks/run_all_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Does a COVID-19 X-ray classifier look at the lungs?

This notebook reproduces every result in the [project README](https://github.com/anupbagale/covid-xray-shortcut-learning): a dataset audit, three models, Grad-CAM heatmaps and the final report. The results from the original run are already in the repo's [`results`](https://github.com/anupbagale/covid-xray-shortcut-learning/tree/main/results) folder, so you only need to run this to check them or to experiment.

**What it does**
1. **Audits the dataset** (a few minutes, no GPU needed): can file metadata alone predict the diagnosis, and are any test images duplicated in training?
2. **Trains three models** on a GPU: the original v1 CNN, a pretrained DenseNet121, and the same DenseNet121 trained with the lungs blacked out.
3. **Writes the report** and lets you download the results.

**How to run it**
1. **Runtime → Change runtime type → T4 GPU** (free on Colab).
2. **Runtime → Run all.** The data downloads from Kaggle; if it asks you to log in, see section 3, "Get the data".
3. Keep the tab open. A full run takes about 1–2 hours.

To start over from scratch, use **Runtime → Disconnect and delete runtime** before **Run all**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime -> Change runtime type -> T4 GPU'

## 2. Get the code

In [ ]:
import os, sys
REPO_URL = "https://github.com/anupbagale/covid-xray-shortcut-learning.git"

# First run: download the code. Running again in the same session: update it to
# the latest version on GitHub instead (a second clone would fail and leave old code).
%cd /content
if os.path.isdir("project/.git"):
    !git -C project pull -q
else:
    !git clone -q $REPO_URL project
%cd /content/project
!pip install -q .
!git log -1 --format='Code version: %h %s (%cr)'

# Make the package importable in this running notebook without a restart
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))

## 3. Get the data

Default: download the Kaggle dataset **Chest X-ray (Covid-19 & Pneumonia)** (`prashant268/chest-xray-covid19-pneumonia`) with `kagglehub`. If Kaggle asks you to log in, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets (key icon on the left), or use the Google Drive option below.

The v1 notebooks used this layout and these counts: train 5,144 images, test 1,292.

In [ ]:
USE_DRIVE = False  # set True to use a copy you already have in Google Drive
DRIVE_PATH = "/content/drive/MyDrive/archive (1)/Data"  # the path the v1 notebooks used

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = DRIVE_PATH
else:
    import os, kagglehub
    try:
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass
    except ImportError:
        pass
    DATA_ROOT = kagglehub.dataset_download("prashant268/chest-xray-covid19-pneumonia")

from cxrshortcut.data import index_dataset, counts_table
df = index_dataset(DATA_ROOT)
print(DATA_ROOT)
print(counts_table(df))
print("train:", (df.split == "train").sum(), " test:", (df.split == "test").sum())

## 4. Audit the dataset (no deep learning)

Two questions before training anything:
- Can a classifier that sees **only file metadata** (size, format, colour mode, brightness) predict the diagnosis? If yes, the classes came from different sources, and a CNN can learn the source.
- Are **test images duplicated** in the training set?

In [ ]:
!python scripts/audit_dataset.py --data-root "$DATA_ROOT" --out results/audit

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("results/audit/near_duplicates_examples.png"):
    display(Image("results/audit/near_duplicates_examples.png", width=500))
else:
    print("No near-duplicates found.")

## 5. Train and test the models

Each run trains on `train`, picks the best epoch on a 15% validation split taken from `train`, and only then touches `test`.

**5a. The original v1 CNN**, trained properly this time. `--eval-occluded` also tests it on images with the lungs blacked out.

In [ ]:
!python scripts/run_experiment.py --data-root "$DATA_ROOT" --model original_cnn --epochs 25 --eval-occluded

**5b. DenseNet121** pretrained on ImageNet: 10 epochs training the new head, then 10 epochs fine-tuning the last dense block.

In [ ]:
!python scripts/run_experiment.py --data-root "$DATA_ROOT" --model densenet121 --epochs 10 --fine-tune-epochs 10 --eval-occluded

**5c. The shortcut test.** The same DenseNet121, but trained *and* tested with the lung region blacked out. It can only use what is outside the lungs: borders, corners, text markers, body shape. If it still scores well above chance, the labels can be predicted without looking at the lungs.

In [ ]:
!python scripts/run_experiment.py --data-root "$DATA_ROOT" --model densenet121 --name densenet121_occluded --epochs 10 --fine-tune-epochs 10 --occlude

## 6. Report

In [ ]:
!python scripts/make_report.py --results results
from IPython.display import Markdown
display(Markdown(open("results/summary.md").read()))

In [ ]:
import os
from IPython.display import Image, display
for name in ("original_cnn", "densenet121", "densenet121_occluded"):
    path = f"results/{name}/gradcam_examples.png"
    if os.path.exists(path):
        print(name)
        display(Image(path, width=900))

## 7. Save the results

This zips everything except the trained model files (which are large). Download `results.zip`, then either commit the `results/` folder to the repo or send the zip back for the README write-up.

In [ ]:
!zip -qr results.zip results -x "*.keras"
!ls -lh results.zip
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    pass